# Hebrew Codenames — cross-model semantic alignment probe

Codenames as a *measurement instrument*: a one-word clue transduces a target set into a guess.
We compare two semantic systems on that channel —

- **Encoders** (the geometry): cosine over Hebrew word embeddings — `fasttext` (static baseline), `neodictabert`, `embeddinggemma`, `qwen3-embed`.
- **LLM** (the intent): DictaLM 3.0 giving clues / ranking guesses in natural Hebrew.

**Board** = the real 573-word Hebrew שם-קוד deck. **Clues** = a large frequency-filtered Hebrew vocabulary (never the board), per Koyyalagunta et al. 2021.

> **What this measures.** Agreement between a clue-giver and guesser that *share* an embedding is trivially high; **cross-system agreement measures alignment ("cooperation"), not clue quality** (Koyyalagunta et al. 2021, critiquing Kim et al. 2019). The LLM stands in as the *human-like intent* reference (Kumar et al. 2021: distributional cosine under-predicts human association). **The divergences are the finding.**

**Directions:** `LLM → Encoder` (intent recovery) · `Encoder → LLM` (geometry legibility). **Headline** = Spearman ρ between the encoder's cosine ordering and the LLM's ordering of the board, averaged over rounds.

In [ ]:
import time, json, random, itertools
import numpy as np
from codenames import probe
from codenames.probe import (make_encoder, HebrewLLM, sample_board, encoder_rank, encoder_spymaster,
                   llm_spymaster, llm_guess_ranking, spearman, recovery_at_k,
                   load_clue_vocab, ENCODERS, LLM_FAST, LLM_BIG)

SEED         = 7
N_BOARDS     = 6                 # bump once it's working; LLM time scales with this
ENCODER_KEYS = ["fasttext", "neodictabert", "embeddinggemma", "qwen3-embed"]
CLUE_VOCAB_N = 12000             # size of the frequency-filtered clue vocabulary
LLM_MODEL    = LLM_FAST          # -> LLM_BIG for the 12B quality run (~13GB)

rng = random.Random(SEED)
boards = [sample_board(rng) for _ in range(N_BOARDS)]
clue_vocab = load_clue_vocab(CLUE_VOCAB_N)
print(f"{N_BOARDS} boards | encoders={ENCODER_KEYS} | clue_vocab={len(clue_vocab)} | llm={LLM_MODEL}")

## 1 · Load encoders + precompute clue-vocab embeddings (one-time; the BERT encoders take a few min)

In [ ]:
encoders, clue_emb = {}, {}
for k in ENCODER_KEYS:
    t = time.time(); encoders[k] = make_encoder(k)
    clue_emb[k] = encoders[k].embed(clue_vocab)        # cached so spymaster is fast per board
    print(f"{k:14s} loaded+embedded {len(clue_vocab)} clue words in {time.time()-t:5.1f}s  dim={clue_emb[k].shape[1]}")

## 2 · Encoder-only: what clue does each encoder's geometry give? (no LLM)

In [ ]:
b = boards[0]
print("MY      :", "  ".join(b.my))
print("ASSASSIN:", b.assassin, "\n")
for k, e in encoders.items():
    c = encoder_spymaster(e, b, clue_vocab, clue_emb[k])
    print(f"{k:14s} clue={c.word!r:10} g={c.margin:+.3f} assassin_sim={c.assassin_sim:+.3f}  intended={c.intended}")

## 3 · Encoder ↔ encoder agreement (do the geometries even agree with each other?)

In [ ]:
agg = {p: [] for p in itertools.combinations(ENCODER_KEYS, 2)}
for b in boards:
    clue = encoder_spymaster(encoders[ENCODER_KEYS[0]], b, clue_vocab, clue_emb[ENCODER_KEYS[0]]).word
    orders = {k: encoder_rank(encoders[k], b, clue)[0] for k in ENCODER_KEYS}
    for a, c in itertools.combinations(ENCODER_KEYS, 2):
        agg[(a, c)].append(spearman(orders[a], orders[c]))
print("Inter-encoder board-ranking agreement (Spearman, mean over boards):")
for (a, c), v in agg.items():
    print(f"  {a:14s} vs {c:14s}: {np.mean(v):+.3f}")

## 4 · Load the Hebrew LLM (DictaLM 3.0 via MLX)

In [ ]:
import os; os.environ.setdefault("HF_HUB_OFFLINE", "1")   # use cache; avoids a re-resolve hang
t = time.time(); llm = HebrewLLM(LLM_MODEL)
print(f"LLM loaded in {time.time()-t:.1f}s")

## 5 · Direction A — LLM → Encoder (intent recovery)
LLM is spymaster and *names its targets*; each encoder guesses by nearest-neighbour to the clue.

In [ ]:
rowsA = []
for i, b in enumerate(boards):
    c = llm_spymaster(llm, b)
    if not c or not c.intended:
        print(f"board {i}: clue parse failed -> {c}"); continue
    print(f"board {i}: LLM clue={c.word!r} count={c.count} intended={c.intended}")
    for k, e in encoders.items():
        order, _ = encoder_rank(e, b, c.word)
        rowsA.append(dict(board=i, encoder=k, clue=c.word, k=c.count,
                          recovery=recovery_at_k(order, c.intended, c.count),
                          assassin_topk=(b.assassin in order[:c.count])))
print("\nLLM → Encoder  (mean over rounds):")
for k in ENCODER_KEYS:
    rec = [r['recovery'] for r in rowsA if r['encoder'] == k]
    ass = [r['assassin_topk'] for r in rowsA if r['encoder'] == k]
    if rec:
        print(f"  {k:14s} intent_recovery@k={np.mean(rec):.2f}   assassin_in_topk={np.mean(ass):.2f}")

## 6 · Direction B — Encoder → LLM (geometry legibility) + headline ρ
Encoder is spymaster; the LLM ranks the board for that clue. Headline = Spearman(encoder cosine order, LLM order).

In [ ]:
rowsB = []
for i, b in enumerate(boards):
    for k, e in encoders.items():
        c = encoder_spymaster(e, b, clue_vocab, clue_emb[k])
        enc_order, _ = encoder_rank(e, b, c.word)
        llm_order = llm_guess_ranking(llm, b, c.word)
        rowsB.append(dict(board=i, encoder=k, clue=c.word, k=c.count,
                          spearman=spearman(enc_order, llm_order),
                          llm_recovery=recovery_at_k(llm_order, c.intended, c.count),
                          assassin_rank_llm=llm_order.index(b.assassin)))
print("Encoder → LLM  (mean over rounds):  [headline = spearman]")
for k in ENCODER_KEYS:
    sp = [r['spearman'] for r in rowsB if r['encoder'] == k]
    rc = [r['llm_recovery'] for r in rowsB if r['encoder'] == k]
    ar = [r['assassin_rank_llm'] for r in rowsB if r['encoder'] == k]
    print(f"  {k:14s} spearman={np.mean(sp):+.3f}  llm_recovery={np.mean(rc):.2f}  "
          f"assassin_rank={np.mean(ar):.1f}/25")

## 7 · Disagreement viewer — the round where geometry and LLM split hardest

In [ ]:
worst = min(rowsB, key=lambda r: r['spearman'])
b = boards[worst['board']]; e = encoders[worst['encoder']]
enc_order, sims = encoder_rank(e, b, worst['clue'])
llm_order = llm_guess_ranking(llm, b, worst['clue'])
print(f"encoder={worst['encoder']}  clue={worst['clue']!r}  rho={worst['spearman']:+.3f}\n")
print(f"{'word':12} {'role':9} {'enc#':>4} {'llm#':>4} {'cos':>7}")
for w in sorted(b.words, key=lambda w: enc_order.index(w)):
    print(f"{w:12} {b.role[w]:9} {enc_order.index(w):4d} {llm_order.index(w):4d} {sims[w]:+.3f}")

## 8 · Save

In [ ]:
out = dict(seed=SEED, n_boards=N_BOARDS, encoders=ENCODER_KEYS,
           clue_vocab_n=len(clue_vocab), llm=LLM_MODEL, direction_A=rowsA, direction_B=rowsB)
with open("results.json", "w") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print("saved results.json")